# Missing Values and Range Checks

*Keep absent, malformed, and out-of-range measurements distinct during cleaning.*


## Missing Tokens Are Not Observed Zeros

The source uses `None`, `NA`, and empty text for absent scores. Normalize these
tokens before conversion. A nullable `Int64` column stores whole-number scores
and a missing marker together. Replacing absence with 0 or -1 would change a
mean unless those replacement values were explicitly excluded.


In [1]:
import pandas as pd

raw_scores = pd.Series(
    ["100", None, "NA", "", "None", "eighty", "105", "-3", "86"], dtype="string"
)
tokens = raw_scores.str.strip()
missing_input = tokens.isna() | tokens.str.lower().isin(["", "na", "none"])
numeric_scores = pd.to_numeric(tokens.mask(missing_input), errors="coerce").astype(
    "Int64"
)
format_error = ~missing_input & numeric_scores.isna()
range_error = numeric_scores.notna() & ~numeric_scores.between(0, 100)

report_df = pd.DataFrame(
    {
        "raw_score": raw_scores,
        "numeric_score": numeric_scores,
        "missing_input": missing_input,
        "format_error": format_error,
        "range_error": range_error,
    }
)
print(report_df.to_string(index=False))


raw_score  numeric_score  missing_input  format_error  range_error
      100            100          False         False        False
     <NA>           <NA>           True         False        False
       NA           <NA>           True         False        False
                    <NA>           True         False        False
     None           <NA>           True         False        False
   eighty           <NA>          False          True        False
      105            105          False         False         True
       -3             -3          False         False         True
       86             86          False         False        False


There are four missing inputs, one format error (`eighty`), and two range
errors (105 and -3). The two valid observed scores are 100 and 86.


## Preserve the Reason a Score Is Unavailable

Keep the raw score and flags alongside the analysis value. `.mask()` marks
range errors as missing for this analysis. This decision excludes them from
the mean while retaining their records for correction.


In [2]:
report_df["score"] = numeric_scores.mask(range_error)
print("Analysis scores:", report_df["score"].tolist())
print("Valid score count:", report_df["score"].count())
print("Observed mean:", report_df["score"].mean())
print(
    "Mean after filling unavailable values with zero:",
    round(report_df["score"].fillna(0).mean(), 2),
)


Analysis scores: [100, <NA>, <NA>, <NA>, <NA>, <NA>, <NA>, <NA>, 86]
Valid score count: 2
Observed mean: 93.0
Mean after filling unavailable values with zero: 20.67


The observed mean is 93 from two valid scores. Filling all seven unavailable
scores with zero produces 20.67, a different quantity based on invented values.


## Clipping Is a Policy That Changes Values

`.clip(lower=0, upper=100)` bounds numeric values. This reproduces the source's
clipping operation so its effect is visible. Such a rule may be appropriate for
a bounded display, but it does not discover the true score behind an entry error.
A statistically unusual value inside the valid range can still be genuine.


In [3]:
original_scores = pd.Series([102, -5, 100, 86], name="original_score")
clipped_scores = original_scores.clip(lower=0, upper=100)
print(
    pd.DataFrame(
        {"original_score": original_scores, "clipped_score": clipped_scores}
    ).to_string(index=False)
)
print("Changed values:", int((original_scores != clipped_scores).sum()))


 original_score  clipped_score
            102            100
             -5              0
            100            100
             86             86
Changed values: 2


Two values change: 102 becomes 100 and -5 becomes 0. These are chosen
bounds, not verified corrections. The retained analysis above keeps invalid
grades unavailable instead of converting them to those bounds.


Detect missing inputs before conversion, preserve error reasons, and distinguish a valid extreme observation from a rule violation. Filling and clipping require a stated interpretation.
